---
title: 1a) The Connectome Interpreter Tutorial - Linear Methods
authors:
  - name: Aarushi Vardhan
    affiliations:
      - University of Toronto / University of Cambridge
  - name: Sapolnach Prompiengchai
    affiliations:
      - University of Oxford
---

In [1]:
# 1. load your packages at the top
import pandas as pd
import plotly.express as px
import os
from dotenv import load_dotenv
from neuprint import Client
load_dotenv()
c = Client(os.getenv("NEUPRINT_SERVER"), dataset=os.getenv("NEUPRINT_DATASET"), token=os.getenv("NEUPRINT_TOKEN"))

Your explanation has the right basic idea, but there is one important correction: \(M^2\) describes paths with exactly **two hops**, not one hop. A single hop is represented by \(M\) itself.

I would scaffold the section more gradually and separate three ideas: direct connectivity, a single indirect path, and then matrix multiplication.

Here is a beginner-friendly rewrite you can use:

# What is effective connectivity?

So far, we have looked at **direct connections** between neurons. For example, suppose neuron A connects directly to neuron C:

$$
A \longrightarrow C
$$

We can describe the strength of this connection as the fraction of C's input that comes from A. For example, if A provides 60% of C's input, the connection strength is

$$
A\rightarrow C = 0.60.
$$

We looked at this type of normalisation in [Notebook 1C of the neuPrint tutorial](./6c-neuprint_plotting.ipynb).

But neurons do not only influence neurons to which they are directly connected. Activity can also travel through **intermediate neurons**.

For example:

$$
A \longrightarrow B \longrightarrow C
$$

Here, A does not connect directly to C. Nevertheless, A can influence C through B.

This is the basic idea behind **effective connectivity**: we want to measure how strongly one neuron can influence another through indirect, multi-step paths.

## Following one indirect path

Consider the following pathway:

$$
A \xrightarrow{40\%} B \xrightarrow{60\%} C
$$

The percentages tell us how much of the **target neuron's input** comes from the source neuron.

So:

* A provides 40% of B's input.
* B provides 60% of C's input.

We want to know:

> How much of C's input can be attributed to A through the path \(A\rightarrow B\rightarrow C\)?

We multiply the two fractions:

$$
0.40\times0.60=0.24.
$$

So the effective connection from A to C **along this path** is

$$
A\rightarrow B\rightarrow C = 0.24.
$$

In other words, A accounts for 24% of C's input through this particular two-step path.

### Why do we multiply?

Imagine C receives 100 units of input.

B provides 60% of C's input:

$$
100\times0.60=60.
$$

So 60 of those units come through B.

But only 40% of B's input came from A. Therefore, the amount attributable to A is

$$
60\times0.40=24.
$$

Thus,

$$
\frac{24}{100}=0.24.
$$

This gives us a useful rule:

$$
\boxed{\text{For connections along a path, multiply the connection strengths.}}
$$

<div style="text-align: center;">

:::{figure} ../static/effective_connectivity.webp
:width: 1000px

**Effective connectivity and path compression.** Adapted from [Yijie Yin, 2024](https://connectome-interpreter.readthedocs.io/en/latest/tutorials/matmul.html).
:::

</div>

## Representing direct connections with a matrix

Doing these calculations one neuron at a time would quickly become impractical. Instead, we can represent all the direct connections in a **matrix**.

For the toy circuit in the figure above (left), the connectivity matrix is shown (right)

The **rows** represent the source, or presynaptic, neurons:

$$
\text{row}=\text{{pre}}
$$

and the **columns** represent the target, or postsynaptic, neurons:

$$
\text{column}=\text{{post}}.
$$

Therefore, an entry \(M_{ij}\) means

$$
\boxed{M_{ij}=\text{direct connection from neuron }i\text{ to neuron }j.}
$$

For example,

$$
M_{ab}=0.3
$$

means

$$
a\rightarrow b=0.3.
$$

The first row of the matrix is

$$
[0,\;0.3,\;0.5].
$$

This tells us all the direct connections leaving neuron \(a\):

$$
a\rightarrow a=0,
\qquad
a\rightarrow b=0.3,
\qquad
a\rightarrow c=0.5.
$$

Notice that this matrix contains only **direct, one-hop connections**.

So:

$$
\boxed{M=\text{connectivity after exactly one hop}}
$$

## From one hop to two hops

What if we want to know how strongly \(a\) can influence another neuron after going through **one intermediate neuron**?

That requires two connections, or **two hops**:

$$
a\rightarrow ?\rightarrow \text{target}.
$$

We can calculate all such two-hop connections at once by multiplying the matrix by itself:

$$
M^2=M\times M.
$$

Therefore,

$$
\boxed{M^2=\text{connectivity through exactly two hops}.}
$$

Let's calculate one entry of \(M^2\).

Suppose we want the upper-left entry:

$$
(M^2)_{aa}.
$$

This asks:

$$
\boxed{\text{How strongly can }a\text{ reach }a\text{ in exactly two hops?}}
$$

There are three possible intermediate neurons:

$$
a\rightarrow a\rightarrow a,
$$

$$
a\rightarrow b\rightarrow a,
$$

and

$$
a\rightarrow c\rightarrow a.
$$

We calculate the strength of each path separately.

Through \(a\):

$$
a\xrightarrow{0}a\xrightarrow{0}a
$$

so

$$
0\times0=0.
$$

Through \(b\):

$$
a\xrightarrow{0.3}b\xrightarrow{0.2}a
$$

so

$$
0.3\times0.2=0.06.
$$

Through \(c\):

$$
a\xrightarrow{0.5}c\xrightarrow{0.8}a
$$

so

$$
0.5\times0.8=0.40.
$$

But now there is a new idea: there is **more than one possible path** from \(a\) back to \(a\).

We therefore add the contributions from all possible paths:

$$
0+0.06+0.40=0.46.
$$

Thus,

$$
\boxed{(M^2)_{aa}=0.46.}
$$

This gives us our second rule:

$$
\boxed{\text{Multiply along a path; add across different paths.}}
$$

## Where does matrix multiplication come in?

Matrix multiplication performs exactly the calculation we just did.

To calculate the \(a,a\) entry of \(M^2\), we take **row \(a\)** from the first matrix:

$$
\begin{bmatrix}
0&0.3&0.5
\end{bmatrix}
$$

and **column \(a\)** from the second matrix:

$$
\begin{bmatrix}
0\\
0.2\\
0.8
\end{bmatrix}.
$$

We multiply corresponding entries and add them:

$$
(M^2)_{aa}
=
\begin{bmatrix}
0&0.3&0.5
\end{bmatrix}
\begin{bmatrix}
0\\
0.2\\
0.8
\end{bmatrix}
$$

$$
=0(0)+0.3(0.2)+0.5(0.8)
$$

$$
=0+0.06+0.40
$$

$$
=\boxed{0.46}.
$$

Each multiplication corresponds to **one possible path**:

$$
\underbrace{0(0)}_{a\rightarrow a\rightarrow a}
+
\underbrace{0.3(0.2)}_{a\rightarrow b\rightarrow a}
+
\underbrace{0.5(0.8)}_{a\rightarrow c\rightarrow a}.
$$

This is why matrix multiplication is so useful for connectivity analysis. It automatically finds the contribution of every possible intermediate neuron.

For a real connectome, there may be thousands or millions of possible paths. Rather than calculating them individually, matrix multiplication performs all of these calculations simultaneously.

The same idea extends to longer paths:

$$
\boxed{
\begin{aligned}
M   &:\quad \text{1-hop connectivity}\\
M^2 &:\quad \text{2-hop connectivity}\\
M^3 &:\quad \text{3-hop connectivity}\\
M^k &:\quad \text{\(k\)-hop connectivity}
\end{aligned}}
$$

## Why do the columns sum to 1?

There is one final property of this particular connectivity matrix that is important.

Remember that a **column represents a target neuron**.

For example, column \(a\) contains

$$
\begin{bmatrix}
0\\
0.2\\
0.8
\end{bmatrix}.
$$

This means that neuron \(a\) receives:

$$
0\% \text{ from }a,
\qquad
20\% \text{ from }b,
\qquad
80\% \text{ from }c.
$$

Together,

$$
0+0.2+0.8=1.
$$

Or, in percentages,

$$
0\%+20\%+80\%=100\%.
$$

The same is true for every column:

$$
\boxed{\text{Each column sums to 1 because it represents 100\% of a target neuron's input.}}
$$

This also gives a useful way to remember the orientation of the matrix:

$$
\boxed{\text{rows = where the signal comes from/ source}}
$$

$$
\boxed{\text{columns = where the signal goes to / target}}
$$
